# Volcano Plots — Clinical Outcome DEGs

Differential expression from `pydeseq2`, contrast **5Plus vs 2Minus** clinical outcome
(n = 10 per group). DE statistics are read directly from `adata.var` of
`bulk_counts_clinical_outcome.h5ad`.

Figures are exported as **SVG** (vector) and **PNG @ 600 dpi** to
`outputs/images/clinical_outcome/`.

Run with the **`spatial`** conda env.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from adjustText import adjust_text

# ---- global aesthetics ----
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'svg.fonttype': 'none',        # keep text as text in the SVG
    'pdf.fonttype': 42,
    'axes.linewidth': 1.1,
})

IMG_DIR = f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome'
os.makedirs(IMG_DIR, exist_ok=True)
IMG_DIR

In [ ]:
H5AD = f'{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/clinical_outcome/bulk_counts_clinical_outcome.h5ad'
adata = sc.read_h5ad(H5AD)
adata

In [ ]:
# DE stats live directly in var -> pull them out and keep only tested genes
df = adata.var.copy()
df = df.dropna(subset=['log2FoldChange', 'padj']).copy()
df['neglog10padj'] = -np.log10(df['padj'])
print(f'Tested genes with DE stats: {len(df):,}')
df[['gene_label', 'Symbol', 'log2FoldChange', 'padj', 'neglog10padj']].head()

In [ ]:
# ---- thresholds & significance category ----
LOG2FC_THRESH = 0.5
PADJ_THRESH   = 0.05

df['Significance'] = 'Not significant'
df.loc[(df['log2FoldChange'] >=  LOG2FC_THRESH) & (df['padj'] < PADJ_THRESH), 'Significance'] = 'Upregulated'
df.loc[(df['log2FoldChange'] <= -LOG2FC_THRESH) & (df['padj'] < PADJ_THRESH), 'Significance'] = 'Downregulated'

print(df['Significance'].value_counts())

In [ ]:
# ---- shared palette & helpers ----
# Volcano convention: up (higher in Good) = green, down (higher in Poor) = red, NS = gray.
# These are the project's fixed outcome colours (see heatmap_up_down / GSEA notebooks) so that
# red always means Poor outcome. Direction is additionally encoded positionally (left/right of
# x = 0) and by the "Higher in Poor" / "Higher in Good" headers, so it is never colour-alone --
# which matters because #2c9d2a / #E63946 do not separate well under deuteranopia.
PALETTE = {
    'Upregulated':     '#2c9d2a',   # Good  (5Plus)  -- green
    'Downregulated':   '#E63946',   # Poor  (2Minus) -- red
    'Not significant': '#C2C7CE',   # light gray
}
HL_COLOR = '#F5A623'                 # gold, for ACC hotspot genes


def save_fig(fig, name):
    """Save a figure as both SVG (vector) and PNG @ 600 dpi."""
    for ext in ('svg', 'png'):
        fig.savefig(os.path.join(IMG_DIR, f'{name}.{ext}'),
                    format=ext, dpi=600, bbox_inches='tight',
                    facecolor='white')
    print(f'saved {name}.svg + {name}.png')


# focus the x-range on the regulated genes (+margin) so a lone far outlier
# doesn't stretch the axis and flatten the plot
_reg = df[df['Significance'] != 'Not significant']['log2FoldChange']
_xmax = np.ceil(_reg.abs().max()) + 0.5
XLIM = (-_xmax, _xmax)


def base_volcano(figsize=(10, 9), title='Differentially expressed genes'):
    """Draw the base volcano (points, threshold lines, labels) and return fig, ax."""
    fig, ax = plt.subplots(figsize=figsize)

    # plot NS first (bottom), then regulated on top
    for cat in ['Not significant', 'Downregulated', 'Upregulated']:
        sub = df[df['Significance'] == cat]
        ax.scatter(sub['log2FoldChange'], sub['neglog10padj'],
                   c=PALETTE[cat], s=26,
                   alpha=0.45 if cat == 'Not significant' else 0.8,
                   linewidths=0, label=cat, rasterized=(cat == 'Not significant'))

    # threshold guides
    ax.axhline(-np.log10(PADJ_THRESH), color='0.35', ls='--', lw=1.1, zorder=0)
    ax.axvline( LOG2FC_THRESH, color='0.35', ls='--', lw=1.1, zorder=0)
    ax.axvline(-LOG2FC_THRESH, color='0.35', ls='--', lw=1.1, zorder=0)

    ax.set_xlim(XLIM)
    ax.set_xlabel('log$_2$ fold change', fontsize=16)
    ax.set_ylabel('-log$_{10}$ (adjusted p-value)', fontsize=16)
    ax.tick_params(axis='both', labelsize=12)

    if title:
        ax.set_title(title, fontsize=19, weight='bold', pad=26)
    # outcome direction: +log2FC = higher in Good (5Plus), -log2FC = higher in Poor (2Minus)
    ax.text(0.25, 1.01, 'Higher in Poor', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=13, weight='bold', color='0.25')
    ax.text(0.75, 1.01, 'Higher in Good', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=13, weight='bold', color='0.25')

    # legend just outside the plot so gene labels never collide with it
    leg = ax.legend(title='Regulation', frameon=False, title_fontsize=13,
                    fontsize=12, loc='upper left', bbox_to_anchor=(1.02, 1.0))
    leg.get_title().set_fontweight('bold')
    sns.despine(ax=ax)
    return fig, ax


# genes to keep out of the label layer even though they rank highly (still plotted as points)
LABEL_EXCLUDE = {'LINC01637'}


def named(sub):
    """Drop genes that shouldn't carry a text label.

    `gene_label` falls back to the raw ENSG accession when a gene has no approved
    symbol, which reads as noise on the figure; LABEL_EXCLUDE holds individually
    dropped genes. Filtering here means the top-N picks below skip those and take
    the next gene instead.
    """
    lab = sub['gene_label'].astype(str)
    return sub[~lab.str.startswith('ENSG') & ~lab.isin(LABEL_EXCLUDE)]


def label_genes(ax, genes_df, color, ha_default='left'):
    """Place gene labels and de-overlap them with adjust_text."""
    texts = []
    for _, r in genes_df.iterrows():
        texts.append(ax.text(r['log2FoldChange'], r['neglog10padj'], r['gene_label'],
                             color=color, fontsize=12, weight='bold',
                             ha=ha_default, va='bottom'))
    return texts

## 1. Plain volcano (no labels)

In [ ]:
fig, ax = base_volcano(title='Differentially expressed genes')
save_fig(fig, 'volcano_clinical_plain')
plt.show()

## 2. Labeled volcano — top 5 up / down by significance (padj)

In [ ]:
top_up   = named(df[df['Significance'] == 'Upregulated']).nsmallest(5, 'padj')
top_down = named(df[df['Significance'] == 'Downregulated']).nsmallest(5, 'padj')
print(top_up['gene_label'].tolist(), top_down['gene_label'].tolist())

fig, ax = base_volcano(title='Top DE genes by significance')
# emphasise the labelled points
for grp, col in [(top_up, PALETTE['Upregulated']), (top_down, PALETTE['Downregulated'])]:
    ax.scatter(grp['log2FoldChange'], grp['neglog10padj'], s=55,
               facecolors=col, edgecolors='white', linewidths=0.8, zorder=5)

texts  = label_genes(ax, top_up,   PALETTE['Upregulated'], 'left')
texts += label_genes(ax, top_down, PALETTE['Downregulated'], 'right')
adjust_text(texts, ax=ax, expand=(1.4, 1.8), only_move={'text': 'xy'},
            arrowprops=dict(arrowstyle='-', color='0.5', lw=0.7))
save_fig(fig, 'volcano_clinical_labeled_padj')
plt.show()

## 3. Labeled volcano — top 5 up / down by effect size (|log2FC|, among significant)

In [ ]:
sig = df[df['padj'] < PADJ_THRESH]
top_up_fc   = named(sig[sig['log2FoldChange'] >=  LOG2FC_THRESH]).nlargest(5, 'log2FoldChange')
top_down_fc = named(sig[sig['log2FoldChange'] <= -LOG2FC_THRESH]).nsmallest(5, 'log2FoldChange')
print(top_up_fc['gene_label'].tolist(), top_down_fc['gene_label'].tolist())

fig, ax = base_volcano(title='Top DE genes by effect size')
for grp, col in [(top_up_fc, PALETTE['Upregulated']), (top_down_fc, PALETTE['Downregulated'])]:
    ax.scatter(grp['log2FoldChange'], grp['neglog10padj'], s=55,
               facecolors=col, edgecolors='white', linewidths=0.8, zorder=5)

texts  = label_genes(ax, top_up_fc,   PALETTE['Upregulated'], 'left')
texts += label_genes(ax, top_down_fc, PALETTE['Downregulated'], 'right')
adjust_text(texts, ax=ax, expand=(1.4, 1.8), only_move={'text': 'xy'},
            arrowprops=dict(arrowstyle='-', color='0.5', lw=0.7))
save_fig(fig, 'volcano_clinical_labeled_lfc')
plt.show()

## 4. ACC hotspot highlight

Curated ACC-related genes overlaid as gold diamonds, merging the canonical ACC set with the
newly added significant hits. **Significant** (padj < 0.05): MYC, TP53, FGFR3 (down) and
KDM6A, MUC12 (up). **Canonical but non-significant** in this contrast (marked with `*`):
MYB, NFIB, NOTCH1 (down) and MYBL1 (up). Genes are matched on `Symbol`; the top-5 up/down DE
genes are also labelled for context.

In [ ]:
# Curated ACC-related genes: canonical set + newly added significant hits; see heatmap_acc_genes.ipynb
HOTSPOT = ['MYB', 'NFIB', 'MYC', 'MYBL1', 'NOTCH1', 'TP53', 'FGFR3', 'KDM6A', 'MUC12']
hot = df[df['Symbol'].isin(HOTSPOT)].copy()
# append ' *' to any listed gene that is NOT significant (canonical genes below the threshold)
_sym = hot['Symbol'].astype(str)
hot['label'] = np.where(hot['padj'] >= PADJ_THRESH, _sym + ' *', _sym)
print(hot[['Symbol', 'log2FoldChange', 'padj', 'Significance']].to_string(index=False))

# slightly larger canvas than the other tweaks to fit the merged label set (still < original 10x9)
fig, ax = base_volcano(figsize=(9, 8), title='ACC-related genes')

# top significant genes (context labels, thin)
texts = []
for grp, col, ha in [(top_up, PALETTE['Upregulated'], 'left'),
                     (top_down, PALETTE['Downregulated'], 'right')]:
    for _, r in grp.iterrows():
        texts.append(ax.text(r['log2FoldChange'], r['neglog10padj'], r['gene_label'],
                             color=col, fontsize=9, ha=ha, va='bottom', alpha=0.8))

# hotspot overlay: gold diamonds (shape-encoded so identity isn't colour-alone)
ax.scatter(hot['log2FoldChange'], hot['neglog10padj'], marker='D', s=110,
           facecolors=HL_COLOR, edgecolors='black', linewidths=1.2,
           zorder=6, label='ACC gene (* = ns)')
for _, r in hot.iterrows():
    texts.append(ax.text(r['log2FoldChange'], r['neglog10padj'], r['label'],
                         color='black', fontsize=12, weight='bold',
                         ha='left', va='bottom', zorder=7))

# rebuild legend to include the hotspot entry (outside the plot)
leg = ax.legend(title='Regulation', frameon=False, title_fontsize=12,
                fontsize=11, loc='upper left', bbox_to_anchor=(1.02, 1.0))
leg.get_title().set_fontweight('bold')

adjust_text(texts, ax=ax, expand=(1.3, 1.7), only_move={'text': 'xy'},
            arrowprops=dict(arrowstyle='-', color='0.5', lw=0.7))
save_fig(fig, 'volcano_clinical_acc_highlight')
plt.show()

In [ ]:
# ---- confirm all exports exist ----
for f in sorted(os.listdir(IMG_DIR)):
    if f.startswith('volcano_clinical'):
        print(f)